[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/ml-curriculum/06_rnn/06_rnn.ipynb)

# 06. Recurrent Neural Networks (RNN)

> 원본 강의: [Lec 12, 모두를 위한 머신러닝과 딥러닝](https://hunkim.github.io/ml/) — RNN 기본, Hi Hello RNN, Long Sequence RNN, Stacked RNN, 시계열 RNN

## 이론

### 1) 왜 RNN인가
지금까지 다룬 모델(Linear/Logistic Regression, MLP, CNN)은 입력을 독립적으로 처리합니다. 하지만 문장, 시계열처럼 **순서(sequence)**가 의미를 가지는 데이터에는 "이전 정보를 기억"하는 구조가 필요합니다.

### 2) RNN Cell
매 타임스텝마다 입력 $x_t$와 이전 **은닉 상태(hidden state)** $h_{t-1}$을 함께 받아 새로운 $h_t$를 계산합니다.
$$h_t = \tanh(W_x x_t + W_h h_{t-1} + b)$$
같은 가중치($W_x, W_h$)를 모든 타임스텝에서 **공유**하기 때문에, 임의 길이의 시퀀스를 처리할 수 있습니다.

### 3) 활용 형태
- **many-to-many**: 문자 시퀀스 -> 문자 시퀀스 (예: 다음 문자 예측, "hihell" -> "ihello")
- **many-to-one**: 시퀀스 -> 하나의 값/클래스 (예: 감성분석, 다음 시점 값 예측)
- **Stacked RNN**: RNN 층을 여러 겹 쌓아 더 복잡한 패턴을 학습

### 4) 한계
기본 RNN은 시퀀스가 길어지면 역시 Vanishing/Exploding Gradient 문제를 겪습니다. 실무에서는 LSTM, GRU를 주로 사용합니다 (이번 노트북에서는 원본 강의와 동일하게 기본 RNN 구조를 직접 체험하는 데 집중합니다).

## 실습 0. Colab 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)
if IN_COLAB:
    !pip install -q torch matplotlib numpy

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print("device:", device)

## 실습 1. Char-RNN — "hihello" 다음 글자 예측

원본 강의의 대표 실습입니다. `"hihell"`을 한 글자씩 넣으면서, 각 시점마다 **다음 글자**를 예측하도록 RNN을 학습시켜 `"ihello"`를 맞히게 합니다.

In [ ]:
chars = list("hielo")  # 사용되는 고유 문자
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for i, c in enumerate(chars)}

text = "hihello"
x_str = text[:-1]  # "hihell"
y_str = text[1:]   # "ihello"

x_idx = [char2idx[c] for c in x_str]
y_idx = [char2idx[c] for c in y_str]

n_classes = len(chars)
x_onehot = np.eye(n_classes)[x_idx]  # (seq_len, n_classes)

X = torch.tensor(x_onehot, dtype=torch.float32).unsqueeze(0).to(device)  # (batch=1, seq_len, input_size)
Y = torch.tensor(y_idx, dtype=torch.long).unsqueeze(0).to(device)        # (batch=1, seq_len)

print("입력:", x_str, "-> 목표:", y_str)
print("X shape:", X.shape, "Y shape:", Y.shape)

In [ ]:
class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)          # out: (batch, seq_len, hidden_size)
        return self.fc(out)           # (batch, seq_len, output_size)

model = CharRNN(input_size=n_classes, hidden_size=8, output_size=n_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(100):
    optimizer.zero_grad()
    outputs = model(X)                                  # (1, seq_len, n_classes)
    loss = loss_fn(outputs.view(-1, n_classes), Y.view(-1))
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0 or epoch == 99:
        pred = outputs.argmax(dim=2).squeeze(0).cpu().numpy()
        pred_str = "".join(idx2char[i] for i in pred)
        print(f"epoch {epoch:3d}  loss={loss.item():.4f}  예측='{pred_str}'")

## 실습 2. 시계열 예측 — Sine 곡선의 다음 값 맞히기

원본 강의의 "시계열 RNN" 실습에 해당합니다. 과거 몇 개의 값을 보고 바로 다음 값을 예측하는 many-to-one RNN을 만듭니다.

In [ ]:
t = np.linspace(0, 100, 1000)
series = np.sin(t)

SEQ_LEN = 20

def make_sequences(series, seq_len):
    xs, ys = [], []
    for i in range(len(series) - seq_len):
        xs.append(series[i:i + seq_len])
        ys.append(series[i + seq_len])
    return np.array(xs), np.array(ys)

X_seq, y_seq = make_sequences(series, SEQ_LEN)

split = int(len(X_seq) * 0.8)
X_train = torch.tensor(X_seq[:split], dtype=torch.float32).unsqueeze(-1).to(device)  # (N, seq_len, 1)
y_train = torch.tensor(y_seq[:split], dtype=torch.float32).unsqueeze(-1).to(device)
X_test = torch.tensor(X_seq[split:], dtype=torch.float32).unsqueeze(-1).to(device)
y_test = torch.tensor(y_seq[split:], dtype=torch.float32).unsqueeze(-1).to(device)

X_train.shape, y_train.shape

In [ ]:
class TimeSeriesRNN(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        last_step = out[:, -1, :]   # many-to-one: 마지막 타임스텝의 hidden state만 사용
        return self.fc(last_step)

ts_model = TimeSeriesRNN().to(device)
optimizer = torch.optim.Adam(ts_model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(150):
    optimizer.zero_grad()
    pred = ts_model(X_train)
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()
    if epoch % 30 == 0 or epoch == 149:
        print(f"epoch {epoch:3d}  train_mse={loss.item():.5f}")

In [ ]:
ts_model.eval()
with torch.no_grad():
    test_pred = ts_model(X_test).cpu().numpy().flatten()

plt.figure(figsize=(10, 4))
plt.plot(y_test.cpu().numpy().flatten(), label="실제값")
plt.plot(test_pred, label="RNN 예측값", linestyle="--")
plt.title("Sine 시계열: RNN 예측 vs 실제")
plt.legend()
plt.show()

## 정리 & 연습 문제
- 실습 1에서 `hidden_size`를 2~3으로 줄이면 학습이 잘 안 될 수 있습니다. 왜 그런지 생각해보세요.
- 실습 2에서 `nn.RNN`을 `nn.LSTM`이나 `nn.GRU`로 바꿔서 성능/수렴 속도를 비교해보세요 (인터페이스는 거의 동일합니다).
- 여기까지 진행하면 원본 강의 시즌 1(딥러닝 기본)의 핵심 흐름을 한 바퀴 돈 것입니다. 이후에는 시즌 RL(강화학습)이나 NLP로 확장할 수 있습니다.


**해설/정답**: [06_rnn_solutions.ipynb](06_rnn_solutions.ipynb)
